лекция: Дистрибуция кода и развертывание
Практикум по Python‑пакетам (**egg, sdist, wheel**), безопасности цепочки поставки (**supply‑chain**), работе с **APT** в Docker, лучшим практикам **Dockerfile**, и созданию **Kubernetes‑ready** приложений.

---
Лекция ориентирована на инженеров DevOps / MLOps, которые хотят:
- понимать форматы дистрибуции кода в Python
- собирать reproducible пакеты (sdist/wheel, deb)
- строить безопасные образы Docker и минимизировать атаки на цепочку поставки
- готовить приложения к развёртыванию в Kubernetes

Мы будем чередовать **объяснения → код → схемы → практикум**.


## План лекции
1. Введение и мотивация
2. Форматы дистрибуции Python: egg → sdist → wheel
3. Supply‑chain атаки и защита
4. APT в Docker
5. Лучшие практики Dockerfile
6. Kubernetes‑ready приложения
7. Debian `.deb` пакеты
8. Лабораторные работы и упражнения
9. Заключение

## 1. Введение: зачем всё это
На продакшн‑системах важно:
- иметь **повторяемые сборки** и предсказуемые артефакты
- минимизировать уязвимости и риски supply‑chain атак
- создавать образы контейнеров, соответствующие best‑practice
- чтобы приложение было готово к Kubernetes: probes, graceful shutdown, ресурсы, безопасность

Типичный pipeline: разработчик → пакет (wheel) → CI/CD → контейнер → Kubernetes.


### ASCII‑схема жизненного цикла
```mermaid
graph TD
A[Код Python] --> B[sdist + wheel]
B --> C[CI/CD pipeline]
C --> D[Docker multi-stage build]
D --> E[Образ в Registry]
E --> F[Kubernetes Pod]
```


## 2. Форматы дистрибуции Python: egg, sdist, wheel
### История и мотивация
- **egg** — устаревший формат из эпохи `easy_install`. Не даёт воспроизводимости.
- **sdist** (source distribution) — исходный код и метаданные. Создаётся командой `python -m build --sdist`.
- **wheel** — бинарный формат (PEP 427), позволяет установить пакет без компиляции.

**Best practice**: всегда публиковать *и* sdist, *и* wheel. Устанавливать wheel.


### Структура минимального проекта
```text
awesome_demo/
  pyproject.toml
  README.md
  src/awesome_demo/__init__.py
  src/awesome_demo/cli.py
```

**pyproject.toml:**
```toml
[build-system]
requires = ["setuptools>=69", "wheel"]
build-backend = "setuptools.build_meta"

[project]
name = "awesome-demo"
version = "0.1.0"
description = "Пример пакета"
readme = "README.md"
requires-python = ">=3.9"

[project.scripts]
awesome-demo = "awesome_demo.cli:main"
```


### Сборка и установка
```bash
python -m pip install --upgrade pip build
python -m build .                 # создаст dist/*.tar.gz и dist/*.whl
pip install dist/*.whl            # установка wheel
```


## 3. Supply‑chain атаки: примеры и защита
**Типы атак:**
- *Typosquatting* — злоумышленник публикует пакет `reqeusts` вместо `requests`
- *Dependency confusion* — внутренний пакет перехватывается внешней публикацией с таким же именем
- Вредоносный код в `setup.py` при установке sdist

### Схема атаки (упрощённо)
```mermaid
graph LR
A[Dev/CI] -->|pip install foo| B[PyPI]
B -->|вместо корпоративного foo| C[Malicious foo]
C --> D[Заражённая среда]
```


### Меры защиты
- Фиксировать версии и хэши: `pip-compile --generate-hashes`
- Использовать внутренний индекс/зеркало PyPI
- Запретить implicit‑updates (`--require-hashes`)
- Автоматическая проверка зависимостей: `pip-audit`, `safety`
- Использовать PEP 517/518 (`pyproject.toml`) вместо `setup.py`


In [ ]:
%%bash
# пример генерации lock-файла с хэшами
pipx run pip-tools pip-compile pyproject.toml --generate-hashes -o requirements.lock


## 4. APT в Docker
Частые ошибки:
- Отдельный RUN для `apt-get update` и `install` → cache‑issue
- Неочищенные `/var/lib/apt/lists/*` → увеличенный размер
- Использование устаревшего `apt-key`

### Best practice сниппет
```Dockerfile
RUN apt-get update \
 && apt-get install -y --no-install-recommends ca-certificates curl gnupg \
 && install -d -m 0755 /etc/apt/keyrings \
 && curl -fsSL https://download.docker.com/linux/debian/gpg | gpg --dearmor -o /etc/apt/keyrings/docker.gpg \
 && chmod a+r /etc/apt/keyrings/docker.gpg \
 && echo "deb [arch=$(dpkg --print-architecture) signed-by=/etc/apt/keyrings/docker.gpg] https://download.docker.com/linux/debian $(. /etc/os-release && echo $VERSION_CODENAME) stable" > /etc/apt/sources.list.d/docker.list \
 && apt-get update \
 && apt-get install -y --no-install-recommends docker-ce-cli \
 && rm -rf /var/lib/apt/lists/*
```


## 5. Лучшие практики Dockerfile
### Основные приёмы:
- **Multi‑stage build**: разделение стадии сборки и рантайма
- **Non‑root user**: `USER appuser`
- **Slim base**: `python:3.12‑slim`
- Чистить кеши `apt`, `pip --no-cache-dir`


### Пример multi‑stage Dockerfile
```Dockerfile
FROM python:3.12-slim AS build
RUN apt-get update && apt-get install -y --no-install-recommends build-essential && rm -rf /var/lib/apt/lists/*
WORKDIR /work
COPY awesome_demo/ ./awesome_demo/
RUN pip install -U pip build && python -m build awesome_demo -o /dist

FROM python:3.12-slim
RUN useradd -m appuser
USER appuser
WORKDIR /app
COPY --from=build /dist/*.whl /tmp/
RUN pip install --no-cache-dir /tmp/*.whl && rm -rf /tmp/*.whl
ENTRYPOINT ["awesome-demo"]
```


## 6. Docker compose

Инструмент, который умеет запускать сразу несколько контейнеров Docker, настроенных для совместной работы, причём всё конфигурируется одним файлом. Он читается как руководство: как сконфигурировать и запустить отдельные контейнеры. Запускается всё одной командой.

Структура проекта описывается в YAML-файле под названием docker-compose.yml. 
В нём можно указать:

    - список сервисов (приложения, базы данных и т.д.),
    - зависимости между ними,
    - сетевые настройки,
    - переменные окружения,
    - тома и порты.

### пример docker-compose.yml
```
version: '3.8'

services:
  app:
    build: .
    ports:
      - "8000:8000"
    env_file:
      - .env
    depends_on:
      - postgres
      - redis
    volumes:
      - ./app:/app
    restart: unless-stopped

  postgres:
    image: postgres:15-alpine
    environment:
      POSTGRES_DB: fastapi_db
      POSTGRES_USER: user
      POSTGRES_PASSWORD: password
    ports:
      - "5432:5432"
    volumes:
      - postgres_data:/var/lib/postgresql/data
    restart: unless-stopped

  redis:
    image: redis:7-alpine
    ports:
      - "6379:6379"
    volumes:
      - redis_data:/data
    restart: unless-stopped

  nginx:
    image: nginx:alpine
    ports:
      - "80:80"
    volumes:
      - ./nginx.conf:/etc/nginx/nginx.conf
    depends_on:
      - app
    restart: unless-stopped

volumes:
  postgres_data:
  redis_data:
```

## 7. Kubernetes‑ready приложения
**Чек‑лист:**
- health‑probes: readiness, liveness, startup
- graceful shutdown: SIGTERM → preStop → закрыть соединения → grace period
- stateless, конфиг через ENV / ConfigMap / Secret
- ресурсы (requests/limits), non‑root, readOnlyRootFilesystem
- логи stdout/stderr, метрики Prometheus


### Пример Deployment
```yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: awesome-demo
spec:
  replicas: 2
  selector:
    matchLabels: { app: awesome-demo }
  template:
    metadata:
      labels: { app: awesome-demo }
    spec:
      terminationGracePeriodSeconds: 30
      containers:
      - name: app
        image: ghcr.io/your-org/awesome-demo:1.0.0
        ports:
          - containerPort: 8080
        readinessProbe:
          httpGet: { path: /healthz/ready, port: 8080 }
          initialDelaySeconds: 3
        livenessProbe:
          httpGet: { path: /healthz/live,  port: 8080 }
          initialDelaySeconds: 10
        startupProbe:
          httpGet: { path: /healthz/startup, port: 8080 }
          failureThreshold: 30
          periodSeconds: 2
        resources:
          requests: { cpu: "100m", memory: "128Mi" }
          limits:   { cpu: "500m", memory: "512Mi" }
        securityContext:
          runAsNonRoot: true
          readOnlyRootFilesystem: true
```


## 8. Debian `.deb` пакет
### Минимальная структура
```text
pkgroot/
  DEBIAN/control
  usr/local/bin/awesome-demo
```
```bash
dpkg-deb --build pkgroot awesome-demo_0.1.0_all.deb
sudo apt install ./awesome-demo_0.1.0_all.deb
```


### Утилиты для упаковки в deb пакет:
- dh-python:
    https://github.com/p1otr/dh-python
- dh-virtualenv:
    https://github.com/spotify/dh-virtualenv
- stdeb
    https://github.com/stdeb/stdeb

```text
pkgroot/
  DEBIAN/control
  usr/local/bin/awesome-demo
```
```bash
dpkg-deb --build pkgroot awesome-demo_0.1.0_all.deb
sudo apt install ./awesome-demo_0.1.0_all.deb
```


## 9. Лабораторные задания
1. Соберите wheel‑пакет для `awesome_demo`
2. Постройте multi‑stage Docker‑образ и запустите его
3. Создайте `.deb` и установите локально
4. Разверните Deployment в kind/minikube и протестируйте probes
5. Запустите `pip-audit` на своём lock‑файле


# 10. Заключение
Мы прошли путь от исходного кода до развёрнутого приложения в Kubernetes.
Ключевые идеи:
- reproducible builds (wheel, deb)
- защита цепочки поставки
- secure и минимальные Docker‑образы
- k8s‑готовность приложений
